# Phase 4C — Scaling benchmark (1K → 20K documents)

**Instructor-aligned question:** Does search scale linearly as corpus size grows? Where does it slow down, and what might happen with 10× more data?

This is a **CPU, in-memory search+top-k ranking benchmark** using the existing 20K real MiniLM embeddings and Phase 4A's fitted TruncatedSVD k=100 index. No new model training or parameter search.

Methods (same document IDs and same 100 queries):
1. **Exact sequential BLAS (384d):** one query at a time, reference deployment pattern.
2. **Exact batched BLAS (384d):** 100 queries in one matrix multiply, selected fast Phase 3 strategy.
3. **Reduced batched BLAS (100d):** same batch and ranking on the existing SVD index.

Measure search **including top-10 ranking** (not loading data, query encoding, or fitting SVD), record warmup separately, repeat **5 times per method per N** for full run, archive all raw measurements and medians. Use 1 BLAS thread for reproducibility. A batch's latency and throughput are both reported; **do not confuse per-batch time with interactive single-query latency**.

Pilot uses 1K/5K and 20 queries with 2 repeats to catch errors; only FULL is reportable. Exact vs SVD ranking quality was assessed in Phase 4B, not repeated here.

In [ ]:
%pip -q install numpy pandas matplotlib scikit-learn threadpoolctl
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,time,json,platform
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits

ROOT=Path('/content/drive/MyDrive/HP-SS-Engine')
EMB=ROOT/'phase2_matrix_ops/2A_sparse_dense/qwen_20k_minilm_384_float32.npy'
OUT=ROOT/'phase4_final_system'
PILOT=True  # First run pilot, then change to False and RUN ALL cells.
DOC_SIZES=[1000,5000] if PILOT else [1000,5000,10000,20000]
NUM_QUERIES=20 if PILOT else 100
REPEATS=2 if PILOT else 5
WARMUPS=1 if PILOT else 2
BLAS_THREADS=1
TOP_K=10
SEED=20261021
TAG='pilot' if PILOT else 'full'
for p in (OUT/'benchmarks',OUT/'figures'):p.mkdir(parents=True,exist_ok=True)
print(dict(mode=TAG,doc_sizes=DOC_SIZES,queries=NUM_QUERIES,repeats=REPEATS,warmups=WARMUPS,blas_threads=BLAS_THREADS))

## 1. Load real 384d and 100d indexes; choose identical queries

The full 20K SVD index is already fitted and saved by Phase 4A. For smaller N, use its **first N rows** (no refitting); this isolates the cost of searching a larger index. All query vectors are drawn deterministically from the 20K embedding corpus; query IDs are fixed across all N, even if some queries refer to documents outside the smaller index. The reduced queries use the **same fitted full-20K SVD model**, transformed and L2-normalized.

Query encoding and model fitting are intentionally excluded from measured search latency; their cost should be disclosed separately in the final report.

In [ ]:
assert EMB.exists(),f'Missing {EMB}'
E=np.load(EMB,mmap_mode='r')
assert E.shape==(20000,384) and E.dtype==np.float32,(E.shape,E.dtype)
svd_model=OUT/'models/truncated_svd_k100_full_20k.joblib'
svd_index=OUT/'models/reduced_index_k100_full_20k.npy'
assert svd_model.exists() and svd_index.exists(),'Run Phase 4A FULL first.'
import joblib
svd=joblib.load(svd_model)
Z_full=np.load(svd_index,mmap_mode='r')
assert Z_full.shape==(20000,100) and Z_full.dtype==np.float32
rng=np.random.default_rng(SEED)
query_ids=rng.choice(20000,size=NUM_QUERIES,replace=False)
Q_raw=np.asarray(E[query_ids],dtype=np.float32)
Q384=np.ascontiguousarray(normalize(Q_raw,axis=1),dtype=np.float32)
Q100=np.ascontiguousarray(normalize(svd.transform(Q_raw),axis=1),dtype=np.float32)
assert Q384.shape==(NUM_QUERIES,384) and Q100.shape==(NUM_QUERIES,100)
assert np.isfinite(Q384).all() and np.isfinite(Q100).all()
print('Query sample:',query_ids[:8].tolist())
print('Q384',Q384.shape,'Q100',Q100.shape,'full index',Z_full.shape)

## 2. Define three real search+ranking methods

The deterministic top-k ranking uses decreasing cosine score with smaller document ID as a tie-breaker. Each method returns a matrix of Top-10 document IDs. The timed region includes all dot products and all rankings. All arrays are float32 and contiguous. We avoid accidentally timing data loading or normalization.

In [ ]:
def topk_ids(scores,k=TOP_K):
    # Full deterministic ranking; identical across methods.
    return np.lexsort((np.arange(scores.shape[0]),-scores))[:k]

def sequential_exact(X,Q):
    output=np.empty((len(Q),TOP_K),dtype=np.int32)
    for i,q in enumerate(Q):
        output[i]=topk_ids(X@q)
    return output

def batched_search(X,Q):
    scores=X@Q.T
    output=np.empty((len(Q),TOP_K),dtype=np.int32)
    for i in range(len(Q)):
        output[i]=topk_ids(scores[:,i])
    return output

# Correctness sanity check outside benchmark: exact sequential and batched
# may differ in floating-point rounding. Count differences, don't conceal them.
with threadpool_limits(limits=BLAS_THREADS):
    X_check=np.ascontiguousarray(normalize(np.asarray(E[:1000],dtype=np.float32),axis=1),dtype=np.float32)
    A=sequential_exact(X_check,Q384[:min(10,NUM_QUERIES)])
    B=batched_search(X_check,Q384[:min(10,NUM_QUERIES)])
    set_matches=sum(set(a)==set(b) for a,b in zip(A,B))
    order_matches=sum(np.array_equal(a,b) for a,b in zip(A,B))
print(f'Preflight exact Top-10 set matches: {set_matches}/{len(A)}, ordered matches: {order_matches}/{len(A)}')
if set_matches<len(A):print('NOTE: inspect numerical near ties if exact methods differ; Phase 4B contains full correctness analysis.')

## 3. Run scaling benchmark

For each N, create the exact and reduced **indexes before timing**. Each method gets warmup runs, then 5 measured runs in FULL mode. Alternate method order between N sizes to reduce systematic ordering bias. Time the complete 100-query batch, including top-k ranking. Archive per-run wall-clock time, milliseconds per query (batch average), throughput, and memory of each index. Do not label the average as interactive query latency.

In [ ]:
methods=['exact_sequential_384d','exact_batched_384d','svd_batched_100d']
rows=[]
environment={'python':platform.python_version(),'numpy':np.__version__,'cpu_count':os.cpu_count(),'platform':platform.platform(),'blas_threads':BLAS_THREADS,'seed':SEED,'queries':NUM_QUERIES,'repeats':REPEATS,'warmups':WARMUPS,'doc_sizes':DOC_SIZES}
with threadpool_limits(limits=BLAS_THREADS):
    for size_idx,N in enumerate(DOC_SIZES):
        X=np.ascontiguousarray(normalize(np.asarray(E[:N],dtype=np.float32),axis=1),dtype=np.float32)
        Z=np.ascontiguousarray(Z_full[:N],dtype=np.float32)
        assert X.shape==(N,384) and Z.shape==(N,100)
        assert np.allclose(np.linalg.norm(Z,axis=1),1.0,atol=1e-5)
        order=methods[size_idx%len(methods):]+methods[:size_idx%len(methods)]
        print(f'\nN={N:,}; exact={X.nbytes/1024**2:.2f} MiB; reduced={Z.nbytes/1024**2:.2f} MiB')
        for name in order:
            matrix,queries=(Z,Q100) if name=='svd_batched_100d' else (X,Q384)
            fn=sequential_exact if name=='exact_sequential_384d' else batched_search
            for _ in range(WARMUPS):
                _=fn(matrix,queries)
            for rep in range(1,REPEATS+1):
                start=time.perf_counter()
                ids=fn(matrix,queries)
                ms=(time.perf_counter()-start)*1000
                assert ids.shape==(NUM_QUERIES,TOP_K) and np.all((ids>=0)&(ids<N))
                rows.append(dict(mode=TAG,n_docs=N,method=name,dim=matrix.shape[1],
                    n_queries=NUM_QUERIES,repeat=rep,batch_ms=ms,
                    avg_ms_per_query_in_batch=ms/NUM_QUERIES,
                    queries_per_second=NUM_QUERIES*1000/ms,
                    index_mib=matrix.nbytes/(1024**2),warmups=WARMUPS,blas_threads=BLAS_THREADS))
            med=np.median([r['batch_ms'] for r in rows if r['n_docs']==N and r['method']==name])
            print(f'  {name}: median {med:.3f} ms per {NUM_QUERIES}-query batch')
raw=pd.DataFrame(rows)
assert len(raw)==len(DOC_SIZES)*len(methods)*REPEATS
raw_path=OUT/'benchmarks'/f'phase4c_scaling_raw_{TAG}.csv'
raw.to_csv(raw_path,index=False)
env_path=OUT/'benchmarks'/f'phase4c_environment_{TAG}.json'
env_path.write_text(json.dumps(environment,indent=2),encoding='utf-8')
print('\nSaved:',raw_path,env_path)

## 4. Summaries, scaling ratios, and plots

We use the **median of the 5 repeated batch timings**, and also report min/max. Speedup at a fixed N is sequential 384d median divided by another method's median. Growth ratio is each method's N=20K time divided by its N=1K time. A ratio near 20 for 20× documents suggests approximately linear growth *over this measured range*; fixed overheads, cache effects and sorting may change this. **Do not extrapolate 200K as a measured result.**

In [ ]:
summary=raw.groupby(['n_docs','method','dim','n_queries','index_mib'],as_index=False).agg(
    median_batch_ms=('batch_ms','median'),min_batch_ms=('batch_ms','min'),
    max_batch_ms=('batch_ms','max'),mean_batch_ms=('batch_ms','mean'),
    sd_batch_ms=('batch_ms','std'))
summary['median_avg_ms_per_query_in_batch']=summary.median_batch_ms/summary.n_queries
summary['median_queries_per_second']=summary.n_queries*1000/summary.median_batch_ms
baseline=summary[summary.method=='exact_sequential_384d'].set_index('n_docs').median_batch_ms
summary['speedup_vs_sequential_exact']=summary.apply(lambda r:float(baseline.loc[r.n_docs]/r.median_batch_ms),axis=1)
first=summary[summary.n_docs==min(DOC_SIZES)].set_index('method').median_batch_ms
summary['growth_vs_1k']=summary.apply(lambda r:float(r.median_batch_ms/first.loc[r.method]),axis=1)
summary_path=OUT/'benchmarks'/f'phase4c_scaling_summary_{TAG}.csv'
summary.to_csv(summary_path,index=False)
print(summary[['n_docs','method','median_batch_ms','median_queries_per_second','speedup_vs_sequential_exact','growth_vs_1k']].to_string(index=False))
print('Saved:',summary_path)

fig,ax=plt.subplots(figsize=(9,5))
for name,g in summary.groupby('method',sort=False):
    g=g.sort_values('n_docs')
    ax.plot(g.n_docs,g.median_batch_ms,marker='o',label=name)
ax.set(xlabel='Number of documents',ylabel=f'Median search+ranking time for {NUM_QUERIES} queries (ms)',
       title=f'Phase 4C — CPU search scaling ({TAG}, {REPEATS} repeats)')
ax.grid(alpha=.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig_path=OUT/'figures'/f'phase4c_scaling_{TAG}.png'
fig.savefig(fig_path,dpi=180)
plt.show()
print('Saved:',fig_path)

## 5. How to interpret and report the FULL run

1. Verify **four N values × three methods × five repeats = 60 raw timing rows**, all on the same 100 query IDs, and that warmups occurred.
2. Compare median times, throughput, and speedup of exact sequential, exact batched, and SVD batched at 20K.
3. Compare 1K→20K time growth with 20× corpus growth; identify deviations from linear scaling. Sorting/top-k overhead and matrix multiplication both contribute.
4. **Memory**: report raw exact vs reduced index MiB; do not mistake this for total process RAM.
5. **Limitations**: in-corpus embedding queries, CPU only, one BLAS thread, warmed indexes, no disk I/O, no MiniLM encoding or SVD fit in the timed region. Batched query throughput is not individual interactive latency.
6. **10× corpus** (200K): give only a reasoned projection and state it was **not measured**.
7. Archive full CSV/PNG in GitHub and update the final findings after reviewing Drive artifacts. Pilot files stay separate.

Next: Phase 4D final interactive demo and report integration, following the original assignment plan.